In [7]:
#Lab 3.2 -  CNN Architectures: LeNet, VGG, ResNet
#1 Import
import torchvision.models as models

import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Feature Extraction Part (Convolutions)
        self.features = nn.Sequential(
            # Input: 1 channel (Grayscale), Output: 32 filters, 3x3 kernel
            nn.Conv2d(1, 32, kernel_size=3, padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2), # Reduces 28x28 to 14x14
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)  # Reduces 14x14 to 7x7
        )
        
        # Classification Part (Fully Connected)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            # 64 channels * 7 * 7 spatial dimension = 3136 input features
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10) # 10 classes for FashionMNIST
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [8]:
#2 Inspect Architectures
from torchvision import models
import torch

#Define Instance 
lenet = SimpleCNN()

#Define Standard Models
vgg = models.vgg11(weights=None)
resnet = models.resnet18(weights=None)

#Optional: Fix VGG/ResNet to accept 1-channel FashionMNIST images
#This changes the first convolution layer from 3 input channels to 1

vgg.features[0] = torch.nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1)
resnet.conv1 = torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

#Print parameters 
print("LeNet params:", sum(p.numel() for p in lenet.parameters()))
print("VGG params:", sum(p.numel() for p in vgg.parameters()))
print("ResNet params:", sum(p.numel() for p in resnet.parameters()))

LeNet params: 421642
VGG params: 132862184
ResNet params: 11683240


In [ ]:
#3 Train small ResNet on CIFAR-10
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# 1. Setup Data
tfm = transforms.Compose([
    transforms.Resize(224), 
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)) # Standard CIFAR10 normalization
])

train_c = datasets.CIFAR10('data', train=True, download=True, transform=tfm)
test_c = datasets.CIFAR10('data', train=False, download=True, transform=tfm)
train_dl = DataLoader(train_c, batch_size=64, shuffle=True, num_workers=2) # num_workers speeds up resizing
test_dl = DataLoader(test_c, batch_size=128)

# 2. Setup Model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
resnet = models.resnet18(num_classes=10).to(device)
opt = torch.optim.Adam(resnet.parameters(), lr=1e-3)

# 3. Training Loop
for ep in range(2):
    resnet.train()
    total_loss = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        
        opt.zero_grad()
        out = resnet(x)
        loss = F.cross_entropy(out, y)
        loss.backward()
        opt.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {ep+1} completed. Avg Loss: {total_loss/len(train_dl):.4f}")

 17%|█████████████▏                                                                | 28.8M/170M [03:01<31:45, 74.4kB/s]

In [ ]:
#4 Evaluate
def accuracy(model, dl):
    model.eval()
    correct = 0
    total = 0
    # Automatically find the device the model is on (cuda or cpu)
    device = next(model.parameters()).device 
    
    with torch.no_grad():
        for x, y in dl:
            # Move batch to the correct device
            x, y = x.to(device), y.to(device)
            
            out = model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
            
    return correct / total

# Now call it
print(f"ResNet CIFAR10 accuracy: {accuracy(resnet, test_dl):.4f}")